# Mech Training AI — Train Engine Parts Detector (GPU)

Train a YOLOv8 model to detect 50 engine/vehicle components with high accuracy.

**This notebook trains for 80 epochs on a free Google GPU (T4) — takes about 30-60 minutes instead of 10+ hours on CPU.**

### Setup Instructions:
1. Click **Runtime → Change runtime type → T4 GPU**
2. Paste your Roboflow API key in Cell 2 below
3. Click **Runtime → Run all** (or Ctrl+F9)
4. Wait for training to finish (~30-60 min)
5. Download the `engine_parts_best.pt` file when prompted
6. Copy it to `C:\Mech Training\models\engine_parts_best.pt`
7. Run `python main.py --auto --cam 2` to test!

### Getting a Roboflow API Key (free):
1. Go to [app.roboflow.com](https://app.roboflow.com) and sign up
2. Go to **Settings → API Keys**
3. Copy your **Private API Key**

In [ ]:
# Step 1: Install packages and verify GPU
!pip install ultralytics roboflow -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    print("Training on CPU will be VERY slow (10+ hours vs 30-60 min on GPU)")

In [ ]:
# Step 2: PASTE YOUR ROBOFLOW API KEY HERE
API_KEY = ""  # <-- paste your key between the quotes

In [ ]:
# Step 3: Download engine parts dataset from Roboflow
import os
from roboflow import Roboflow

assert API_KEY, "You must paste your Roboflow API key in the cell above!"

rf = Roboflow(api_key=API_KEY)

# Datasets to try (in order of preference)
datasets_to_try = [
    ("computervision-nyq7i", "car-components-dataset-otfbq", "Car Components (best quality)"),
    ("final-year-project-5vbtb", "engine-parts-detector", "Engine Parts Detector"),
    ("team-data", "car-parts-ybiev", "Car Parts (large dataset)"),
]

dataset = None
dataset_name = ""

for workspace_id, project_id, name in datasets_to_try:
    print(f"\nTrying: {name}...")
    try:
        project = rf.workspace(workspace_id).project(project_id)
        # Try versions 1-5
        for v in [2, 1, 3, 4, 5]:
            try:
                version = project.version(v)
                dataset = version.download("yolov8")
                dataset_name = name
                print(f"  Downloaded: {name} (version {v})")
                break
            except Exception as e:
                print(f"  Version {v}: {e}")
                continue
        if dataset:
            break
    except Exception as e:
        print(f"  Could not access: {e}")
        continue

if not dataset:
    raise RuntimeError(
        "Could not download any dataset.\n"
        "Check your API key at: app.roboflow.com > Settings > API Keys"
    )

# Show dataset info
data_yaml = os.path.join(dataset.location, "data.yaml")
print(f"\n{'='*50}")
print(f"  Dataset: {dataset_name}")
print(f"  Location: {dataset.location}")
print(f"{'='*50}")
with open(data_yaml, 'r') as f:
    print(f.read())

In [ ]:
# Step 4: Train YOLOv8n — 80 epochs on GPU
# This should take about 30-60 minutes on a T4 GPU
from ultralytics import YOLO
import torch

# Load pre-trained YOLOv8n as starting point (transfer learning)
model = YOLO("yolov8n.pt")

# Use GPU if available, otherwise CPU
device = "0" if torch.cuda.is_available() else "cpu"
batch_size = 16 if torch.cuda.is_available() else 8

print(f"Training on: {'GPU' if device == '0' else 'CPU'}")
print(f"Batch size: {batch_size}")
print(f"Epochs: 80")
print(f"Image size: 640px")
print(f"\nTraining starting... this will take ~30-60 min on GPU.\n")

# Train!
results = model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=batch_size,
    name="engine_parts",
    patience=20,           # Stop early if no improvement for 20 epochs
    save=True,
    exist_ok=True,
    device=device,
    workers=2,
    verbose=True,
    augment=True,          # Data augmentation for better generalization
    mosaic=1.0,            # Mosaic augmentation
    flipud=0.1,            # Vertical flip (some parts look similar upside down)
    fliplr=0.5,            # Horizontal flip
)

print("\n" + "=" * 50)
print("  TRAINING COMPLETE!")
print("=" * 50)

In [ ]:
# Step 5: Evaluate the model
import glob

# Find the best weights
best_path = "runs/detect/engine_parts/weights/best.pt"
if not os.path.exists(best_path):
    # Try to find it
    candidates = glob.glob("runs/detect/engine_parts*/weights/best.pt")
    if candidates:
        best_path = candidates[-1]  # Latest

print(f"Best model: {best_path}")
best_model = YOLO(best_path)

# Show class names
print(f"\nModel classes ({len(best_model.names)}):")
for i, name in best_model.names.items():
    print(f"  {i}: {name}")

# Run validation
print(f"\nRunning validation...")
metrics = best_model.val()
print(f"\n{'='*50}")
print(f"  mAP50:    {metrics.box.map50:.3f}")
print(f"  mAP50-95: {metrics.box.map:.3f}")
print(f"{'='*50}")

In [ ]:
# Step 6: Download the trained model to your computer
import shutil
from google.colab import files

# Copy best weights
dst = "engine_parts_best.pt"
shutil.copy2(best_path, dst)
file_size = os.path.getsize(dst) / (1024 * 1024)

print(f"Model file: {dst} ({file_size:.1f} MB)")
print(f"\nAfter downloading, copy this file to:")
print(f"  C:\\Mech Training\\models\\engine_parts_best.pt")
print(f"\nThen test with:")
print(f"  python main.py --auto --cam 2")
print(f"\nDownloading now...")

# Trigger browser download
files.download(dst)